# Isomap Manifold Analysis


In [ ]:
import os
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt

from sklearn import manifold
from scipy.spatial.transform import Rotation as R
from scipy.stats import pearsonr


def optimize_isomap_with_results(X_train, n_neighbors_range, pos_data, n_components=2):

    best_model = None
    best_n_neighbors = None
    highest_similarity = -1

    reconstruction_errors = []
    similarity_scores = []
    correlation_curves = {}
    embeddings = []
    actual_n_neighbors = []

    for n_neighbors in n_neighbors_range:
        print(f"Testing Isomap with {n_neighbors} neighbors...")

        model = manifold.Isomap(
            n_neighbors=n_neighbors,
            n_components=n_components
        )

        model.fit(X_train)

        embedding = model.transform(X_train)
        error = model.reconstruction_error()

        reconstruction_errors.append(error)
        actual_n_neighbors.append(n_neighbors)

        print(f"Reconstruction Error: {error:.4f}")

        similarity, correlation_data = compute_similarity_score_with_plot(
            embedding,
            pos_data
        )

        similarity_scores.append(similarity)
        correlation_curves[n_neighbors] = correlation_data
        embeddings.append(embedding)

        if similarity > highest_similarity:
            highest_similarity = similarity
            best_model = model
            best_n_neighbors = n_neighbors

            print(
                f"New best model found: "
                f"NN={n_neighbors}, Similarity={similarity:.4f}"
            )

    return (
        best_model,
        embeddings,
        similarity_scores,
        reconstruction_errors,
        actual_n_neighbors,
        best_n_neighbors,
        correlation_curves
    )


def compute_similarity_score_with_plot(embedding, pos_data):

    max_similarity = -1
    correlation_data = []

    for angle in np.arange(0, 360, 5):
        rotation_matrix = R.from_euler(
            "z",
            angle,
            degrees=True
        ).as_matrix()[:2, :2]

        rotated_embedding = embedding @ rotation_matrix.T

        corr_x, _ = pearsonr(rotated_embedding[:, 0], pos_data[:, 0])
        corr_y, _ = pearsonr(rotated_embedding[:, 1], pos_data[:, 1])

        similarity = (corr_x + corr_y) / 2

        correlation_data.append((angle, corr_x, corr_y, similarity))
        max_similarity = max(max_similarity, similarity)

    return max_similarity, correlation_data


def plot_isomap_results(
    embeddings,
    similarity_scores,
    reconstruction_errors,
    n_neighbors_list,
    position_color
):

    n_plots = len(embeddings)
    fig, axes = plt.subplots(1, n_plots, figsize=(4 * n_plots, 4))

    if n_plots == 1:
        axes = [axes]

    for ax, embedding, score, nn in zip(
        axes,
        embeddings,
        similarity_scores,
        n_neighbors_list
    ):
        ax.scatter(
            embedding[:, 0],
            embedding[:, 1],
            c=position_color,
            s=5
        )

        ax.set_title(f"NN={nn}\nSimilarity={score:.2f}")
        ax.axis("off")

    plt.tight_layout()
    return fig


def plot_correlation_vs_angle_multiple(nn_correlation_curves):

    n_plots = len(nn_correlation_curves)
    fig, axes = plt.subplots(1, n_plots, figsize=(4 * n_plots, 4))

    if n_plots == 1:
        axes = [axes]

    for ax, (nn, data) in zip(axes, nn_correlation_curves.items()):
        angles, corr_x, corr_y, similarities = zip(*data)
        best_angle = angles[np.argmax(similarities)]

        ax.plot(angles, corr_x, label="Corr X", color="blue")
        ax.plot(angles, corr_y, label="Corr Y", color="orange")
        ax.axvline(
            best_angle,
            linestyle="--",
            color="black",
            label=f"Best angle ({best_angle} deg)"
        )

        ax.set_title(f"NN={nn}")
        ax.set_xlabel("Rotation angle (deg)")
        ax.set_ylabel("Correlation")
        ax.legend()

    plt.tight_layout()
    return fig


def plot_arena_with_bins(pos_data, position_color):

    fig = plt.figure(figsize=(6, 6))

    plt.scatter(
        pos_data[:, 0],
        pos_data[:, 1],
        c=position_color,
        s=10
    )

    plt.xlabel("X Position")
    plt.ylabel("Y Position")
    plt.title("Original Arena with Color Coding")
    plt.grid()

    return fig


def save_plot(fig, filename):

    output_dir = os.path.dirname(filename)
    os.makedirs(output_dir, exist_ok=True)

    fig.savefig(f"{filename}.jpg", format="jpg", dpi=300)

    plt.close(fig)


# === Main execution workflow for sample data ===

base_dir = os.getcwd()

input_folder = os.path.join(base_dir, "git_folder")
output_folder = os.path.join(base_dir, "output")

os.makedirs(output_folder, exist_ok=True)

mat_file_path = os.path.join(
    input_folder,
    "SPK_BHV_isomap_input_posBIN.mat"
)

data = sio.loadmat(mat_file_path)

spike_matrix = data["spike_rates_norm_zscore"]
pos_data = np.vstack((
    data["binned_pos_x"].flatten(),
    data["binned_pos_y"].flatten()
)).T

vel_data = data["binned_velocity"].flatten()
position_bin = data["position_bin"].flatten()
position_color = data["position_color"]

# Filter high-velocity epochs
high_vel_indices = np.where(vel_data > 2)[0]

X_train = spike_matrix[:, high_vel_indices].T
pos_data_high_vel = pos_data[high_vel_indices]
position_bin_high_vel = position_bin[high_vel_indices]
position_color_high_vel = position_color[high_vel_indices]

# Optimize Isomap
n_neighbors_list = [10, 50, 90]

(
    best_model,
    embeddings,
    similarity_scores,
    reconstruction_errors,
    actual_n_neighbors,
    best_n_neighbors,
    nn_correlation_curves
) = optimize_isomap_with_results(
    X_train,
    n_neighbors_list,
    pos_data_high_vel
)

best_idx = actual_n_neighbors.index(best_n_neighbors)

best_reconstruction_error = reconstruction_errors[best_idx]
best_similarity_score = similarity_scores[best_idx]
embedding_full = best_model.transform(X_train)

baseline_similarity, full_correlation_data = compute_similarity_score_with_plot(
    embedding_full,
    pos_data_high_vel
)

fig = plot_isomap_results(
    embeddings,
    similarity_scores,
    reconstruction_errors,
    actual_n_neighbors,
    position_color_high_vel
)
save_plot(fig, os.path.join(output_folder, "isomap_results"))

fig = plot_correlation_vs_angle_multiple(nn_correlation_curves)
save_plot(fig, os.path.join(output_folder, "correlation_vs_rotation_angle"))

fig = plot_arena_with_bins(
    pos_data_high_vel,
    position_color_high_vel
)
save_plot(fig, os.path.join(output_folder, "original_arena"))

print(f"Processing completed. Results saved to: {output_folder}")